In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Pas de GPU")

True
Tesla T4


In [2]:
!pip install -q ultralytics pydicom

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.5 MB/s eta 0:00:00a 0:00:01


In [3]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)

/kaggle/input
/kaggle/input/competitions
/kaggle/input/competitions/rsna-pneumonia-detection-challenge
/kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_train_images
/kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_test_images


In [7]:
import os
print(os.listdir('/kaggle/input'))

['competitions']


In [8]:
import os
print(os.listdir('/kaggle/input/competitions'))

['rsna-pneumonia-detection-challenge']


In [9]:
import pandas as pd
labels = pd.read_csv('/kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_train_labels.csv')
print(len(labels))
labels.head()

30227


,patientId,x,y,width,height,Target
0,0004cfab-14fd-4e49-80ba-63a80b6bddd6,NaN,NaN,NaN,NaN,0
1,00313ee0-9eaa-42f4-b0ab-c148ed3241cd,NaN,NaN,NaN,NaN,0
2,00322d4d-1c29-4943-afc9-b6754be640eb,NaN,NaN,NaN,NaN,0
3,003d8fa0-6bf1-40ed-b54c-ac657f8495c5,NaN,NaN,NaN,NaN,0
4,00436515-870c-4b36-a041-de91049b9ab4,264.0,152.0,213.0,379.0,1


In [12]:
for split in ['train', 'test']:
    os.makedirs(f'/kaggle/working/dataset/images/{split}', exist_ok=True)
    os.makedirs(f'/kaggle/working/dataset/labels/{split}', exist_ok=True)

In [13]:
from sklearn.model_selection import train_test_split

grouped = labels.groupby('patientId')
patient_ids = labels['patientId'].unique()
train_ids, test_ids = train_test_split(patient_ids, test_size=0.2, random_state=42)

In [14]:
import pydicom
import numpy as np
from PIL import Image

def convertir_patient(patient_id, split):
    dicom_path = f'/kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_train_images/{patient_id}.dcm'
    dicom = pydicom.dcmread(dicom_path)
    img = dicom.pixel_array
    img = ((img - img.min()) / (img.max() - img.min()) * 255).astype(np.uint8)
    Image.fromarray(img).save(f'/kaggle/working/dataset/images/{split}/{patient_id}.png')

    h, w = img.shape
    boxes = grouped.get_group(patient_id)
    lignes_annotation = []
    for _, row in boxes.iterrows():
        if row['Target'] == 1:
            x, y, bw, bh = row['x'], row['y'], row['width'], row['height']
            x1, y1 = x / w, y / h
            x2, y2 = (x + bw) / w, y / h
            x3, y3 = (x + bw) / w, (y + bh) / h
            x4, y4 = x / w, (y + bh) / h
            lignes_annotation.append(f"0 {x1:.6f} {y1:.6f} {x2:.6f} {y2:.6f} {x3:.6f} {y3:.6f} {x4:.6f} {y4:.6f}")
    with open(f'/kaggle/working/dataset/labels/{split}/{patient_id}.txt', 'w') as f:
        f.write('\n'.join(lignes_annotation))

In [15]:
import random
random.seed(42)
echantillon_train = random.sample(list(train_ids), 200)
echantillon_test = random.sample(list(test_ids), 50)

for pid in echantillon_train:
    convertir_patient(pid, 'train')
for pid in echantillon_test:
    convertir_patient(pid, 'test')
print("Conversion terminee")

Conversion terminee


In [16]:
data_yaml = """
train: /kaggle/working/dataset/images/train
val: /kaggle/working/dataset/images/test

nc: 1
names: ['pneumonie']
"""
with open('/kaggle/working/dataset/data.yaml', 'w') as f:
    f.write(data_yaml)
print(data_yaml)


train: /kaggle/working/dataset/images/train
val: /kaggle/working/dataset/images/test

nc: 1
names: ['pneumonie']



In [17]:
import os
print("Images train:", len(os.listdir('/kaggle/working/dataset/images/train')))
print("Labels train:", len(os.listdir('/kaggle/working/dataset/labels/train')))
print("Images test:", len(os.listdir('/kaggle/working/dataset/images/test')))

Images train: 200
Labels train: 200
Images test: 50


In [18]:
from ultralytics import YOLO

model = YOLO('yolov8m-seg.pt')
results = model.train(
    data='/kaggle/working/dataset/data.yaml',
    epochs=15,
    imgsz=512,
    batch=16,
    project='/kaggle/working/pneumoscan_runs',
    name='pneumoscan_yolov8m_seg',
    patience=5,
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.104 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript

In [19]:
from IPython.display import FileLink
FileLink('/kaggle/working/pneumoscan_runs/pneumoscan_yolov8m_seg/weights/best.pt')

/kaggle/working/pneumoscan_runs/pneumoscan_yolov8m_seg/weights/best.pt

In [22]:
from IPython.display import FileLink
FileLink('/kaggle/working/pneumoscan_runs/pneumoscan_yolov8m_seg/weights/best.pt')

/kaggle/working/pneumoscan_runs/pneumoscan_yolov8m_seg/weights/best.pt